In [0]:
#%pip install yfinance
%pip install TA-Lib

In [0]:

from analisis_precios.data.downloader import FinancialDataDownloader
from analisis_precios.visualization.plotter import FinancialPlotter
from datetime import datetime, timedelta

end_date = datetime.today()
start_date = end_date - timedelta(days=365*4)

extractor = FinancialDataDownloader('AAPL')
df_apple = extractor.download_data(start_date=start_date.strftime('%Y-%m-%d'),
                                   end_date=end_date.strftime('%Y-%m-%d'))

plotter = FinancialPlotter(df_apple, ticker='AAPL')
plotter.plot_price_series(title="Apple Prices - Last 6 Months")

display(df_apple)

In [0]:
df_apple

In [0]:
import talib

df_apple['SMA_21'] = talib.SMA(df_apple['Close'], timeperiod=21)
df_apple['SMA_63'] = talib.SMA(df_apple['Close'], timeperiod=63)

display(df_apple)

In [0]:
df_apple['signal'] = None

cross_up = (df_apple['SMA_21'] > df_apple['SMA_63']) & (df_apple['SMA_21'].shift(1) <= df_apple['SMA_63'].shift(1))
cross_down = (df_apple['SMA_21'] < df_apple['SMA_63']) & (df_apple['SMA_21'].shift(1) >= df_apple['SMA_63'].shift(1))

df_apple.loc[cross_up, 'signal'] = 'BUY'
df_apple.loc[cross_down, 'signal'] = 'SELL'

df_apple

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
plt.plot(df_apple.index, df_apple['Close'], label='Close', color='black', linewidth=2)
plt.plot(df_apple.index, df_apple['SMA_21'], label='SMA 21', color='blue')
plt.plot(df_apple.index, df_apple['SMA_63'], label='SMA 63', color='orange')

sell_idx = df_apple[df_apple['signal'] == 'SELL'].index
sell_prices = df_apple.loc[sell_idx, 'Close']
buy_idx = df_apple[df_apple['signal'] == 'BUY'].index
buy_prices = df_apple.loc[buy_idx, 'Close']

plt.scatter(sell_idx, sell_prices, color='red', marker='o', label='SELL', s=60, zorder=5)
plt.scatter(buy_idx, buy_prices, color='green', marker='o', label='BUY', s=60, zorder=5)

plt.legend()
plt.title('Apple CLOSE, SMA_21, SMA_63, BUY/SELL Signals')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
import numpy as np

# Columna de posición: la posición persiste hasta que cambie la señal
df_apple['position'] = df_apple['signal'].map({'BUY': 1, 'SELL': -1}).replace({None: np.nan}).ffill().fillna(0)

# Columna de crecimiento logarítmico del precio
df_apple['log_return'] = np.log(df_apple['Close'] / df_apple['Close'].shift(1))

display(df_apple)

In [0]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(df_apple.index, df_apple['Close'], label='Close', color='black', linewidth=2)
ax1.plot(df_apple.index, df_apple['SMA_21'], label='SMA 21', color='blue')
ax1.plot(df_apple.index, df_apple['SMA_63'], label='SMA 63', color='orange')

sell_idx = df_apple[df_apple['signal'] == 'SELL'].index
sell_prices = df_apple.loc[sell_idx, 'Close']
buy_idx = df_apple[df_apple['signal'] == 'BUY'].index
buy_prices = df_apple.loc[buy_idx, 'Close']

ax1.scatter(sell_idx, sell_prices, color='red', marker='o', label='SELL', s=60, zorder=5)
ax1.scatter(buy_idx, buy_prices, color='green', marker='o', label='BUY', s=60, zorder=5)

ax2 = ax1.twinx()
ax2.plot(df_apple.index, df_apple['position'], label='Position', color='purple', linewidth=1.5, alpha=0.6)
ax2.set_ylabel('Position', color='purple')
ax2.tick_params(axis='y', labelcolor='purple')

fig.legend(loc='upper left', bbox_to_anchor=(0.13, 0.93))
ax1.set_title('Apple CLOSE, SMA_21, SMA_63, BUY/SELL Signals, Position')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price')
ax1.grid(True)
fig.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt

# Encontrar el índice del primer día con señal
first_signal_idx = df_apple[df_apple['signal'].notna()].index[0]
first_signal_price = df_apple.loc[first_signal_idx, 'Close']

# Subset desde la primera señal
df_base = df_apple.loc[first_signal_idx:].copy()

# Base 100 para buy & hold
df_base['bh_growth'] = 100 * df_base['Close'] / first_signal_price

# Base 100 para estrategia sma
df_base['strategy_growth'] = 100 * (1 + df_base['position'] * df_base['log_return']).cumprod()

plt.figure(figsize=(14, 6))
plt.plot(df_base.index, df_base['bh_growth'], label='Buy & Hold', color='gray', linewidth=2)
plt.plot(df_base.index, df_base['strategy_growth'], label='SMA Strategy', color='blue', linewidth=2)
plt.title('Crecimiento Base 100 desde Primera Señal: SMA Strategy vs Buy & Hold')
plt.xlabel('Fecha')
plt.ylabel('Crecimiento (Base 100)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
df_base

In [0]:
comparison_df = df_base[['bh_growth', 'strategy_growth']]

summary = {
    'Buy & Hold Growth (%)': (comparison_df['bh_growth'].iloc[-1] - comparison_df['bh_growth'].iloc[0]) / comparison_df['bh_growth'].iloc[0] * 100,
    'SMA Strategy Growth (%)': (comparison_df['strategy_growth'].iloc[-1] - comparison_df['strategy_growth'].iloc[0]) / comparison_df['strategy_growth'].iloc[0] * 100,
}

import pandas as pd
summary_df = pd.DataFrame([summary])

display(summary_df)